# 04 — VAEP-Style Action Valuation

This notebook extends the project from a spatial **xT baseline** to a supervised **VAEP-style** action valuation framework.

## Objectives

1. Load the `actions` dataset produced in Notebook 02.
2. Build an action-state feature matrix using spatial, temporal, and possession context.
3. Train two probabilistic models:
   - `P(score in next K actions | state)`
   - `P(concede in next K actions | state)`
4. Evaluate discrimination and calibration.
5. Estimate a first-pass **VAEP-style value** for each action:

   \[
   VAEP(action) = P(score) - P(concede)
   \]

6. Produce player-level and team-level valuation summaries.
7. Persist predictions and summaries for downstream reporting.

## Outputs

- `data/processed/actions_vaep.parquet`
- `data/processed/player_vaep_summary.parquet`
- `data/processed/team_vaep_summary.parquet`
- `db/football_value.duckdb`
  - `actions_vaep`
  - `player_vaep_summary`
  - `team_vaep_summary`


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore")


In [ ]:
try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.calibration import calibration_curve
except Exception as e:
    raise ImportError("scikit-learn is required. Install it with: pip install scikit-learn") from e

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except Exception:
    LIGHTGBM_AVAILABLE = False


## 1. Project paths

In [ ]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [PROCESSED_DIR, DB_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"
ACTIONS_PATH = PROCESSED_DIR / "actions.parquet"

ACTIONS_VAEP_PATH = PROCESSED_DIR / "actions_vaep.parquet"
PLAYER_VAEP_PATH = PROCESSED_DIR / "player_vaep_summary.parquet"
TEAM_VAEP_PATH = PROCESSED_DIR / "team_vaep_summary.parquet"

REPO_ROOT, ACTIONS_PATH, DB_PATH


## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    min_actions_per_player: int = 100
    train_split_quantile: float = 0.80
    save_outputs: bool = True
    use_lightgbm: bool = True
    random_state: int = 42

CFG = Config()
CFG


## 3. Load action dataset

Expected input:
- `data/processed/actions.parquet`

This dataset was produced in Notebook 02 and already contains:
- forward-looking labels
- spatial coordinates
- action success indicators
- possession-level state features


In [ ]:
if not ACTIONS_PATH.exists():
    raise FileNotFoundError(
        f"Actions dataset not found at {ACTIONS_PATH}. Run Notebook 02 first."
    )

actions = pd.read_parquet(ACTIONS_PATH)
print(actions.shape)
actions.head()


In [ ]:
required_cols = [
    "match_id", "event_id", "index", "period", "minute", "second",
    "team", "player", "event_type", "play_pattern",
    "x_start", "y_start", "x_end", "y_end",
    "delta_x", "delta_y", "move_length", "delta_goal_distance",
    "is_progressive_action", "under_pressure",
    "time_since_possession_start", "action_number_in_possession",
    "remaining_actions_in_possession",
    "goal_for_next_k_actions", "goal_against_next_k_actions",
]

missing = [c for c in required_cols if c not in actions.columns]
if missing:
    raise ValueError(f"Notebook 02 output is missing required columns: {missing}")

actions["event_type"].value_counts()


## 4. Build modelling dataset

We keep the full on-ball action universe from Notebook 02:

- Pass
- Carry
- Dribble
- Shot

The targets are already available:

- `goal_for_next_k_actions`
- `goal_against_next_k_actions`

We now convert the action table into a supervised learning dataset.


In [ ]:
df = actions.copy()

# Robust binary casting
for target in ["goal_for_next_k_actions", "goal_against_next_k_actions"]:
    df[target] = df[target].fillna(0).astype(int)

df["under_pressure"] = df["under_pressure"].fillna(False).astype(int)
df["is_progressive_action"] = df["is_progressive_action"].fillna(False).astype(int)

# Spatial helper features
goal_x = 120.0
goal_y = 40.0

df["dist_to_goal_start"] = np.sqrt((goal_x - df["x_start"])**2 + (goal_y - df["y_start"])**2)
df["dist_to_goal_end"] = np.sqrt((goal_x - df["x_end"])**2 + (goal_y - df["y_end"])**2)

# Basic shot angle proxy relative to goal centre
df["angle_to_goal_start"] = np.arctan2(np.abs(goal_y - df["y_start"]), np.maximum(goal_x - df["x_start"], 1e-6))
df["angle_to_goal_end"] = np.arctan2(np.abs(goal_y - df["y_end"]), np.maximum(goal_x - df["x_end"], 1e-6))

# Action-type flags
for action_name in ["Pass", "Carry", "Dribble", "Shot"]:
    df[f"is_{action_name.lower()}"] = df["event_type"].eq(action_name).astype(int)

# Match time proxy
df["match_seconds"] = (
    (df["period"].fillna(1).astype(int) - 1) * 45 * 60
    + df["minute"].fillna(0).astype(int) * 60
    + df["second"].fillna(0).astype(int)
)

# Keep a stable ordering for time-based split
df = df.sort_values(["match_id", "index"], kind="mergesort").reset_index(drop=True)

df.head()


## 5. Train / test split

For a first portfolio-ready implementation, we use a **time-aware split** based on the global event order.

This is preferable to a random split because it reduces optimistic leakage across adjacent actions from the same match.


In [ ]:
split_value = df["match_seconds"].quantile(CFG.train_split_quantile)

# More robust split: first by match ordering, then by action ordering inside the match
match_order = (
    df.groupby("match_id")["match_seconds"]
    .min()
    .sort_values()
    .reset_index()
)
n_train_matches = int(len(match_order) * CFG.train_split_quantile)
train_match_ids = set(match_order.iloc[:n_train_matches]["match_id"].tolist())

train_mask = df["match_id"].isin(train_match_ids)
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print("Train matches:", train_df["match_id"].nunique())
print("Test matches: ", test_df["match_id"].nunique())


## 6. Feature specification

In [ ]:
numeric_features = [
    "x_start", "y_start", "x_end", "y_end",
    "delta_x", "delta_y", "move_length", "delta_goal_distance",
    "dist_to_goal_start", "dist_to_goal_end",
    "angle_to_goal_start", "angle_to_goal_end",
    "under_pressure", "is_progressive_action",
    "time_since_possession_start",
    "action_number_in_possession", "remaining_actions_in_possession",
    "match_seconds",
    "is_pass", "is_carry", "is_dribble", "is_shot",
]

categorical_features = [
    "event_type",
    "play_pattern",
]

target_for = "goal_for_next_k_actions"
target_against = "goal_against_next_k_actions"

X_train = train_df[numeric_features + categorical_features].copy()
X_test = test_df[numeric_features + categorical_features].copy()

y_train_for = train_df[target_for].copy()
y_test_for = test_df[target_for].copy()

y_train_against = train_df[target_against].copy()
y_test_against = test_df[target_against].copy()

print("Positive rate (train, score):", round(y_train_for.mean(), 4))
print("Positive rate (test, score): ", round(y_test_for.mean(), 4))
print("Positive rate (train, concede):", round(y_train_against.mean(), 4))
print("Positive rate (test, concede): ", round(y_test_against.mean(), 4))


## 7. Build preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor


## 8. Define modelling functions

In [ ]:
def build_logreg_pipeline():
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=CFG.random_state,
        ))
    ])

def build_lightgbm_pipeline():
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=CFG.random_state,
        class_weight="balanced",
        verbosity=-1,
    )
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

def evaluate_binary_model(model, X_train, y_train, X_test, y_test, label_name: str):
    model.fit(X_train, y_train)

    train_proba = model.predict_proba(X_train)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "label": label_name,
        "train_auc": roc_auc_score(y_train, train_proba) if y_train.nunique() > 1 else np.nan,
        "test_auc": roc_auc_score(y_test, test_proba) if y_test.nunique() > 1 else np.nan,
        "train_logloss": log_loss(y_train, train_proba, labels=[0, 1]),
        "test_logloss": log_loss(y_test, test_proba, labels=[0, 1]),
        "train_brier": brier_score_loss(y_train, train_proba),
        "test_brier": brier_score_loss(y_test, test_proba),
    }
    return model, train_proba, test_proba, metrics


## 9. Train the scoring model

In [ ]:
if CFG.use_lightgbm and LIGHTGBM_AVAILABLE:
    print("Using LightGBM for the scoring model.")
    score_model = build_lightgbm_pipeline()
else:
    print("Using Logistic Regression for the scoring model.")
    score_model = build_logreg_pipeline()

score_model, train_score_proba, test_score_proba, score_metrics = evaluate_binary_model(
    score_model,
    X_train, y_train_for,
    X_test, y_test_for,
    label_name="goal_for_next_k_actions"
)

score_metrics


## 10. Train the conceding model

In [ ]:
if CFG.use_lightgbm and LIGHTGBM_AVAILABLE:
    print("Using LightGBM for the conceding model.")
    concede_model = build_lightgbm_pipeline()
else:
    print("Using Logistic Regression for the conceding model.")
    concede_model = build_logreg_pipeline()

concede_model, train_concede_proba, test_concede_proba, concede_metrics = evaluate_binary_model(
    concede_model,
    X_train, y_train_against,
    X_test, y_test_against,
    label_name="goal_against_next_k_actions"
)

concede_metrics


## 11. Consolidated evaluation

In [ ]:
metrics_df = pd.DataFrame([score_metrics, concede_metrics])
metrics_df


## 12. Calibration plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_true, y_prob, title in [
    (axes[0], y_test_for, test_score_proba, "Calibration — Score Model"),
    (axes[1], y_test_against, test_concede_proba, "Calibration — Concede Model"),
]:
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed frequency")

plt.tight_layout()
plt.show()


## 13. Attach predictions to the action table

In [ ]:
actions_vaep = df.copy()

actions_vaep["p_score_next_k_actions"] = np.nan
actions_vaep["p_concede_next_k_actions"] = np.nan

actions_vaep.loc[train_mask, "p_score_next_k_actions"] = train_score_proba
actions_vaep.loc[test_mask, "p_score_next_k_actions"] = test_score_proba

actions_vaep.loc[train_mask, "p_concede_next_k_actions"] = train_concede_proba
actions_vaep.loc[test_mask, "p_concede_next_k_actions"] = test_concede_proba

actions_vaep["vaep_value"] = (
    actions_vaep["p_score_next_k_actions"] - actions_vaep["p_concede_next_k_actions"]
)

actions_vaep[[
    "event_type", "team", "player",
    "goal_for_next_k_actions", "goal_against_next_k_actions",
    "p_score_next_k_actions", "p_concede_next_k_actions", "vaep_value"
]].head(10)


## 14. Player-level VAEP summary

In [ ]:
player_vaep_summary = (
    actions_vaep.groupby("player", dropna=False)
    .agg(
        n_actions=("event_id", "count"),
        total_vaep=("vaep_value", "sum"),
        mean_vaep=("vaep_value", "mean"),
        total_p_score=("p_score_next_k_actions", "sum"),
        total_p_concede=("p_concede_next_k_actions", "sum"),
        progressive_actions=("is_progressive_action", "sum"),
    )
    .reset_index()
)

player_vaep_summary["vaep_per_100_actions"] = np.where(
    player_vaep_summary["n_actions"] > 0,
    100 * player_vaep_summary["total_vaep"] / player_vaep_summary["n_actions"],
    np.nan
)

player_vaep_summary = player_vaep_summary[
    player_vaep_summary["player"].notna() &
    (player_vaep_summary["n_actions"] >= CFG.min_actions_per_player)
].copy()

player_vaep_summary = player_vaep_summary.sort_values(
    ["total_vaep", "vaep_per_100_actions"],
    ascending=[False, False]
).reset_index(drop=True)

player_vaep_summary.head(20)


## 15. Team-level VAEP summary

In [ ]:
team_vaep_summary = (
    actions_vaep.groupby("team", dropna=False)
    .agg(
        n_actions=("event_id", "count"),
        total_vaep=("vaep_value", "sum"),
        mean_vaep=("vaep_value", "mean"),
        total_p_score=("p_score_next_k_actions", "sum"),
        total_p_concede=("p_concede_next_k_actions", "sum"),
        progressive_actions=("is_progressive_action", "sum"),
    )
    .reset_index()
)

team_vaep_summary["vaep_per_100_actions"] = np.where(
    team_vaep_summary["n_actions"] > 0,
    100 * team_vaep_summary["total_vaep"] / team_vaep_summary["n_actions"],
    np.nan
)

team_vaep_summary = team_vaep_summary.sort_values(
    ["total_vaep", "vaep_per_100_actions"],
    ascending=[False, False]
).reset_index(drop=True)

team_vaep_summary.head(20)


## 16. Distribution of VAEP values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(actions_vaep["vaep_value"].dropna(), bins=50)
ax.set_title("Distribution of VAEP-Style Action Values")
ax.set_xlabel("VAEP value")
ax.set_ylabel("Frequency")
plt.show()


## 17. Top player and team rankings

In [ ]:
player_vaep_summary.head(15)


In [ ]:
team_vaep_summary.head(15)


## 18. Save outputs

In [ ]:
if CFG.save_outputs:
    actions_vaep.to_parquet(ACTIONS_VAEP_PATH, index=False)
    player_vaep_summary.to_parquet(PLAYER_VAEP_PATH, index=False)
    team_vaep_summary.to_parquet(TEAM_VAEP_PATH, index=False)

    try:
        con.close()
    except:
        pass

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE OR REPLACE TABLE actions_vaep AS SELECT * FROM read_parquet(?)", [str(ACTIONS_VAEP_PATH)])
    con.execute("CREATE OR REPLACE TABLE player_vaep_summary AS SELECT * FROM read_parquet(?)", [str(PLAYER_VAEP_PATH)])
    con.execute("CREATE OR REPLACE TABLE team_vaep_summary AS SELECT * FROM read_parquet(?)", [str(TEAM_VAEP_PATH)])
    con.close()

    print(f"Saved action VAEP dataset to:  {ACTIONS_VAEP_PATH}")
    print(f"Saved player summary to:       {PLAYER_VAEP_PATH}")
    print(f"Saved team summary to:         {TEAM_VAEP_PATH}")
    print(f"Updated DuckDB at:             {DB_PATH}")


## 19. Preview from DuckDB

In [ ]:
con = duckdb.connect(str(DB_PATH))

summary_sql = '''
SELECT
    COUNT(*) AS n_actions,
    AVG(p_score_next_k_actions) AS avg_p_score,
    AVG(p_concede_next_k_actions) AS avg_p_concede,
    AVG(vaep_value) AS avg_vaep,
    SUM(vaep_value) AS total_vaep
FROM actions_vaep
'''
preview = con.execute(summary_sql).df()
con.close()

preview


# Conclusions

## Model Performance

Two supervised probabilistic models were trained to estimate short-term scoring dynamics:

* **P(score in next K actions | state)**
* **P(concede in next K actions | state)**

The models were trained on **124,990 actions** and evaluated on **33,124 out-of-sample actions** across unseen matches.

Model discrimination results:

| Model               | Train AUC | Test AUC  |
| ------------------- | --------- | --------- |
| Score probability   | **0.992** | **0.768** |
| Concede probability | **0.999** | **0.782** |

The test AUC values (~0.77–0.78) indicate **strong predictive power given the extremely low event frequency in football data**, where scoring events typically occur in ~1–2% of actions.

These results are consistent with performance levels reported in football analytics research.

---

# Target Distribution

The dataset exhibits the expected class imbalance typical of football event data:

| Target                    | Train     | Test      |
| ------------------------- | --------- | --------- |
| Goal in next K actions    | **1.21%** | **1.24%** |
| Concede in next K actions | **0.30%** | **0.27%** |

This confirms that the dataset reflects realistic football dynamics and validates the use of **class-balanced training strategies**.

---

# VAEP Action Value Layer

Using the predicted probabilities, each action receives a **VAEP-style value**:

VAEP(action) = P(score) − P(concede)

Across the dataset:

* **Total actions analysed:** 158,114
* **Average VAEP value:** 0.1088
* **Total accumulated VAEP:** 17,202.8

The distribution of VAEP values shows the expected pattern:

* large concentration of **neutral actions**
* smaller number of **high positive value actions**
* rare **high negative value actions**

This reflects the fact that most actions in football have **limited immediate impact on goal probability**, while a small subset drives decisive attacking progression.

---

# Player Contribution Insights

The player ranking produced by the model identifies highly creative and progression-oriented players as top contributors.

Top players by total VAEP include:

* **Ronaldinho**
* **Deco**
* **Lionel Messi**
* **Samuel Eto'o**
* **Andrés Iniesta**
* **Xavi Hernández**
* **Neymar**

These players are historically recognized for:

* ball progression
* chance creation
* attacking impact

Their presence at the top of the ranking suggests that the action valuation framework is capturing **meaningful football signal rather than statistical noise**.

---

# Team Tactical Insights

At the team level, the model highlights teams with strong attacking structures and possession-based play.

Top teams by total VAEP include:

* **Barcelona**
* **Belgium**
* **Brazil**
* **England**
* **Croatia**
* **France**

These teams generate a high volume of progressive and attacking actions, which is reflected in their accumulated VAEP values.

---

# Analytical Value of the VAEP Layer

Compared with spatial models such as **Expected Threat (xT)**, the VAEP-style approach captures additional contextual information:

* possession context
* defensive risk
* action type
* temporal dynamics

This allows the model to evaluate actions based on their **impact on match outcomes rather than spatial progression alone**.

As a result, the VAEP framework provides a richer basis for:

* player impact evaluation
* team style analysis
* recruitment analytics
* tactical profiling

---

# Project Significance

This notebook introduces a **supervised action valuation framework built on raw event data**, representing a significant methodological step beyond purely spatial models.

The project now includes a complete football analytics pipeline:

```
StatsBomb event data
        ↓
Event modelling
        ↓
Action dataset
        ↓
Expected Threat (xT)
        ↓
VAEP-style supervised action valuation
```

This structure mirrors modern football analytics workflows used in professional environments.

---

# Next Step

The next stage of the project will focus on **player value analysis and scouting applications**, including:

* comparison between **xT and VAEP contributions**
* player progression profiles
* contribution per 90 minutes
* scouting shortlist generation

These analyses will demonstrate how the action valuation framework can support **data-driven recruitment and tactical evaluation**.
